In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
from math import factorial
from scipy.ndimage import rotate

# Создаем директорию для хранения датасета
dataset_dir = 'Multi_alph_slab' 
if not os.path.exists(dataset_dir):
    os.makedirs(dataset_dir)

# Вспомогательная функция для вычисления радиальной части полиномов Цернике
def R(n, m, r):
    """Функция для расчета радиальной части полинома Цернике"""
    if (n - m) % 2 != 0:
        return np.zeros_like(r)  # Если разность нечетная, результат 0

    radial_sum = np.zeros_like(r)
    for k in range((n - m) // 2 + 1):
        c = (-1) ** k * factorial(n - k) / (factorial(k) * factorial((n + m) // 2 - k) * factorial((n - m) // 2 - k))
        radial_sum += c * r ** (n - 2 * k)
    return radial_sum

# Основная функция для вычисления полиномов Цернике
def zernike_polynomial(n, m, rho, theta):
    """Вычисление полинома Цернике"""
    if m >= 0:
        return R(n, m, rho) * np.cos(m * theta)
    else:
        return R(n, -m, rho) * np.sin(-m * theta)

def noise(img, p):
  rows, cols = img.shape
  noise = (np.random.rand(rows, cols) < p) * np.random.rand(rows, cols) * (np.max(img)/4)
  return img + noise 

# Функция для обрезки изображения до центральной части 128x128
def crop_center(img, crop_size=128):
    h, w = img.shape
    startx = w // 2 - (crop_size // 2)
    starty = h // 2 - (crop_size // 2)    
    return img[starty:starty + crop_size, startx:startx + crop_size]

# Функция для создания аугментаций изображений
def augment_image(img, count):
    augmented_images = []
    for i in range(count):
        # Применение различных аугментаций
        rotated_img = rotate(img, angle=np.random.randint(0, 360), reshape=False)
        noise_img1 = noise(rotated_img, 0.04)
        noise_img2 = noise(rotated_img, 0.08)
        # noise_img = rotated_img + np.random.normal(0, np.random.uniform(0, 1), rotated_img.shape)        
        augmented_images.extend([rotated_img, noise_img1, noise_img2])
    return augmented_images





# Функция для генерации и сохранения изображений
def generate_images(n, m, a_range, class_name, C=75, grid_size=100, num_augmentations=30):
    y, x = np.linspace(-1, 1, grid_size), np.linspace(-1, 1, grid_size)
    alph = [10, 15, 20, 30]
    X, Y = np.meshgrid(x, y)
    R = np.sqrt(X**2 + Y**2)
    theta = np.arctan2(Y, X)
    
    class_dir = os.path.join(dataset_dir, class_name)
    if not os.path.exists(class_dir):
        os.makedirs(class_dir)
    
    image_counter = 0
    for a in a_range:
        Fl = (np.exp(1j * alph[0] * R) * np.exp(1j * C * X + 1j * C * Y) + 
             np.exp(1j * alph[1] * R) * np.exp(1j * (-C) * X + 1j * C * Y) + 
             np.exp(1j * alph[2] * R) * np.exp(1j * C * X + 1j * (-C) * Y) + 
             np.exp(1j * alph[3] * R) * np.exp(1j * (-C) * X + 1j * (-C) * Y))
        # Генерация полинома Цернике
        Z = zernike_polynomial(n, m, R, theta)
        Z1 = np.exp(1j * a * Z)
        Fl = Fl * Z1
        #Z1a = np.cos((a * Z + b * np.sqrt(X ** 2 + Y ** 2)))

        Flf = np.fft.fftshift(np.fft.fft2(Fl, s=[128, 128]))
        
        # Фаза полинома Цернике
        aberration_img = abs(Flf)
        
        # Сохранение изображений
        
        plt.imsave(os.path.join(class_dir, f'zernike_aberration_n{n}_m{m}_a{a:.2f}.png'), aberration_img, cmap='gray')
        
        # # Применение аугментаций и сохранение
        # aberration_augmentations = augment_image(aberration_img, num_augmentations)
        
        # for i, aug_img in enumerate(aberration_augmentations):
        #     plt.imsave(os.path.join(class_dir, f'zernike_aberration_n{n}_m{m}_a{a:.2f}_aug{i}.png'), aug_img, cmap='gray')
                
        image_counter += 1
        if image_counter % 100 == 0:
            print(f'{image_counter} изображений сгенерировано')

# Параметры для генерации
aberration_classes = [
    (2, 2, 'class1'), 
    (3, 1, 'class2'), 
    (3, 3, 'class3'), 
    (4, 2, 'class4'), 
    (4, 4, 'class5'),
    (5, 1, 'class6'), 
    (5, 3, 'class7'),
    (5, 5, 'class8'),
]
a_range_large = np.linspace(1, 10, 3) 

# Генерация изображений для каждого класса
for n, m, class_name in aberration_classes:
    print("class: ", class_name)
    generate_images(n, m, a_range_large, class_name)


class:  class1
class:  class2
class:  class3
class:  class4
class:  class5
class:  class6
class:  class7
class:  class8


: 